# SQL Active Learning + Active Recall Practice Notebook

Use this notebook repeatedly.

- Active recall cards: retrieve concepts from memory
- Auto-checked SQL tasks: write query, verify result
- Score tracking: monitor your practice progress

In [ ]:
from __future__ import annotations

import sqlite3
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)
plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 100)

In [ ]:
# Build practice database
dates = pd.date_range("2024-03-01", periods=45, freq="D")

stations = pd.DataFrame(
    {
        "station_id": [1, 2, 3, 4, 5],
        "station_name": ["Alpha", "Bravo", "Charlie", "Delta", "Echo"],
        "line_code": ["L1", "L1", "L2", "L2", "L3"],
        "region": ["FR", "FR", "FR", "US", "US"],
    }
)

rides_rows = []
row_id = 1
for d in dates:
    for sid in stations["station_id"]:
        base = 900 + sid * 90
        weekend = 0.8 if d.dayofweek >= 5 else 1.0
        riders = int(max(30, base * weekend + rng.normal(0, 70)))
        rides_rows.append((row_id, sid, d.strftime("%Y-%m-%d"), riders))
        row_id += 1
rides = pd.DataFrame(rides_rows, columns=["ride_id", "station_id", "ride_date", "riders"])

weather_rows = []
for d in dates:
    weather_rows.append((d.strftime("%Y-%m-%d"), "FR", round(rng.normal(13, 5), 1), round(max(0, rng.gamma(1.6, 1.4)), 1)))
    weather_rows.append((d.strftime("%Y-%m-%d"), "US", round(rng.normal(15, 6), 1), round(max(0, rng.gamma(1.8, 1.7)), 1)))
weather = pd.DataFrame(weather_rows, columns=["weather_date", "region", "temp_c", "rain_mm"])

conn = sqlite3.connect(":memory:")
stations.to_sql("stations", conn, index=False, if_exists="replace")
rides.to_sql("rides", conn, index=False, if_exists="replace")
weather.to_sql("weather", conn, index=False, if_exists="replace")

def run_sql(q: str) -> pd.DataFrame:
    return pd.read_sql_query(q, conn)

run_sql("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;")

## Part 1 - Active Recall Cards

Start with `reveal=False`, answer mentally, then run with `reveal=True`.

In [ ]:
cards = pd.DataFrame(
    [
        ("What does LEFT JOIN return?", "All rows from left table + matching rows from right table."),
        ("Difference between WHERE and HAVING?", "WHERE filters rows before GROUP BY, HAVING filters groups after aggregation."),
        ("What does the % symbol mean in LIKE?", "Wildcard for zero or more characters."),
        ("What is a CTE?", "Common Table Expression: a named temporary result set with WITH."),
        ("What does ROW_NUMBER() do?", "Assigns sequential row numbers inside a partition/order."),
        ("What is NULL in SQL?", "Unknown/missing value; compare with IS NULL, not '='."),
        ("What is COMMIT?", "Makes current transaction changes permanent."),
        ("What is ROLLBACK?", "Reverts current transaction to previous committed state."),
    ],
    columns=["question", "answer"],
)

def draw_cards(n: int = 5, seed: int = 0, reveal: bool = False) -> pd.DataFrame:
    sample = cards.sample(n=min(n, len(cards)), random_state=seed).reset_index(drop=True)
    if reveal:
        return sample
    out = sample.copy()
    out["answer"] = "(hidden)"
    return out

draw_cards(n=5, seed=1, reveal=False)

In [ ]:
# Reveal mode
draw_cards(n=5, seed=1, reveal=True)

## Part 2 - Auto-checked SQL Exercises

Workflow:

1. Read the exercise list
2. Write your SQL in `my_query_X`
3. Run `check_answer(X, my_query_X)`

In [ ]:
exercises = {
    1: {
        "prompt": "Return first 5 rows from rides with columns ride_id, station_id, riders.",
        "solution": "SELECT ride_id, station_id, riders FROM rides ORDER BY ride_id LIMIT 5;",
    },
    2: {
        "prompt": "Count total rows in rides.",
        "solution": "SELECT COUNT(*) AS total_rows FROM rides;",
    },
    3: {
        "prompt": "Average riders per station_id.",
        "solution": "SELECT station_id, AVG(riders) AS avg_riders FROM rides GROUP BY station_id ORDER BY station_id;",
    },
    4: {
        "prompt": "Join rides + stations and return ride_date, station_name, region, riders (first 10 by ride_id).",
        "solution": """
            SELECT r.ride_date, s.station_name, s.region, r.riders
            FROM rides r
            JOIN stations s ON r.station_id = s.station_id
            ORDER BY r.ride_id
            LIMIT 10;
        """,
    },
    5: {
        "prompt": "Daily total riders by region.",
        "solution": """
            SELECT r.ride_date, s.region, SUM(r.riders) AS total_riders
            FROM rides r
            JOIN stations s ON r.station_id = s.station_id
            GROUP BY r.ride_date, s.region
            ORDER BY r.ride_date, s.region;
        """,
    },
    6: {
        "prompt": "Show only groups where average riders > 1100 (region level).",
        "solution": """
            SELECT s.region, AVG(r.riders) AS avg_riders
            FROM rides r
            JOIN stations s ON r.station_id = s.station_id
            GROUP BY s.region
            HAVING AVG(r.riders) > 1100
            ORDER BY s.region;
        """,
    },
    7: {
        "prompt": "Use a CTE to compute daily total riders, then return top 10 highest-demand days.",
        "solution": """
            WITH daily AS (
                SELECT ride_date, SUM(riders) AS total_riders
                FROM rides
                GROUP BY ride_date
            )
            SELECT *
            FROM daily
            ORDER BY total_riders DESC
            LIMIT 10;
        """,
    },
    8: {
        "prompt": "Use window function to rank days by riders inside each region.",
        "solution": """
            WITH daily AS (
                SELECT r.ride_date, s.region, SUM(r.riders) AS total_riders
                FROM rides r
                JOIN stations s ON r.station_id = s.station_id
                GROUP BY r.ride_date, s.region
            )
            SELECT
                ride_date,
                region,
                total_riders,
                ROW_NUMBER() OVER (PARTITION BY region ORDER BY total_riders DESC) AS demand_rank
            FROM daily
            ORDER BY region, demand_rank;
        """,
    },
}

pd.DataFrame([(k, v["prompt"]) for k, v in exercises.items()], columns=["exercise_id", "prompt"])

In [ ]:
def normalize_df(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    out = df.copy()
    out = out.sort_values(list(out.columns)).reset_index(drop=True)
    return out


def check_answer(exercise_id: int, user_sql: str, verbose: bool = True):
    expected = run_sql(exercises[exercise_id]["solution"])
    got = run_sql(user_sql)

    ok = normalize_df(expected).equals(normalize_df(got))
    if verbose:
        print("Exercise", exercise_id)
        print("Status:", "PASS" if ok else "FAIL")
        if not ok:
            print("Expected sample:")
            display(expected.head())
            print("Your sample:")
            display(got.head())
    return ok

In [ ]:
# Example solved check
my_query_1 = "SELECT ride_id, station_id, riders FROM rides ORDER BY ride_id LIMIT 5;"
check_answer(1, my_query_1)

In [ ]:
# TODO: fill your own queries then run check_answer
my_query_2 = "SELECT COUNT(*) AS total_rows FROM rides;"
my_query_3 = "SELECT station_id, AVG(riders) AS avg_riders FROM rides GROUP BY station_id ORDER BY station_id;"
my_query_4 = """
    SELECT r.ride_date, s.station_name, s.region, r.riders
    FROM rides r
    JOIN stations s ON r.station_id = s.station_id
    ORDER BY r.ride_id
    LIMIT 10;
"""

quick_results = {
    2: check_answer(2, my_query_2, verbose=False),
    3: check_answer(3, my_query_3, verbose=False),
    4: check_answer(4, my_query_4, verbose=False),
}
quick_results

In [ ]:
# Practice score chart
score_df = pd.DataFrame(
    {
        "exercise_id": list(quick_results.keys()),
        "passed": [int(v) for v in quick_results.values()],
    }
)
score_df["failed"] = 1 - score_df["passed"]

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(score_df["exercise_id"].astype(str), score_df["passed"], label="pass")
ax.bar(score_df["exercise_id"].astype(str), score_df["failed"], bottom=score_df["passed"], label="fail")
ax.set_title("Current practice score")
ax.set_xlabel("Exercise")
ax.set_ylabel("Result")
ax.legend()
plt.show()

score_df

## Part 3 - Active Learning Loop

Use this loop every day:

1. 5 recall cards (no reveal)
2. 3 SQL exercises (check yourself)
3. Repeat wrong exercises after 2 hours
4. Weekly: run all exercises and record score trend

In [ ]:
# Optional: confidence tracker (self-assessment)
confidence = pd.DataFrame(
    {
        "topic": ["SELECT/WHERE", "JOIN", "GROUP BY/HAVING", "CTE", "WINDOW FUNCTIONS", "TRANSACTIONS"],
        "confidence_1_to_5": [3, 3, 2, 2, 1, 2],
    }
)

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(confidence["topic"], confidence["confidence_1_to_5"], color="#2ca02c")
ax.set_ylim(0, 5)
ax.set_title("Self-confidence by topic")
ax.tick_params(axis="x", rotation=40)
plt.tight_layout()
plt.show()

confidence